# Language Modeling using Ngram

In this Exercise, we are going to create a bigram language model and its variation. We will build one model for each of the following type and calculate their perplexity:
- Unigram Model
- Bigram Model
- Bigram Model with Laplace smoothing
- Bigram Model with Interpolation
- Bigram Model with Kneser-ney Interpolation

We will also use NLTK which is a natural language processing library for python to make our lives easier.



In [1]:
# #download corpus
!wget --no-check-certificate https://github.com/ekapolc/nlp_2019/raw/master/HW4/BEST2010.zip
!unzip BEST2010.zip

--2025-02-21 13:31:20--  https://github.com/ekapolc/nlp_2019/raw/master/HW4/BEST2010.zip
Resolving github.com (github.com)... 140.82.114.3
Connecting to github.com (github.com)|140.82.114.3|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://raw.githubusercontent.com/ekapolc/nlp_2019/master/HW4/BEST2010.zip [following]
--2025-02-21 13:31:20--  https://raw.githubusercontent.com/ekapolc/nlp_2019/master/HW4/BEST2010.zip
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 7423530 (7.1M) [application/zip]
Saving to: â€˜BEST2010.zipâ€™

BEST2010.zip        100%[===================>]   7.08M  --.-KB/s    in 0.1s    

2025-02-21 13:31:21 (53.9 MB/s) - â€˜BEST2010.zipâ€™ saved [7423530/7423530]

Archive:  BEST2010.zip
   creating: BEST2

In [2]:
!wget https://www.dropbox.com/s/jajdlqnp5h0ywvo/tokenized_wiki_sample.csv

--2025-02-21 13:31:22--  https://www.dropbox.com/s/jajdlqnp5h0ywvo/tokenized_wiki_sample.csv
Resolving www.dropbox.com (www.dropbox.com)... 162.125.5.18, 2620:100:601d:18::a27d:512
Connecting to www.dropbox.com (www.dropbox.com)|162.125.5.18|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://www.dropbox.com/scl/fi/88uzig0mno1b57d6bhwht/tokenized_wiki_sample.csv?rlkey=oya9jw1rljj31jc49fvoaty01 [following]
--2025-02-21 13:31:22--  https://www.dropbox.com/scl/fi/88uzig0mno1b57d6bhwht/tokenized_wiki_sample.csv?rlkey=oya9jw1rljj31jc49fvoaty01
Reusing existing connection to www.dropbox.com:443.
HTTP request sent, awaiting response... 302 Found
Location: https://uca9c5d81c8706f19986c45534c3.dl.dropboxusercontent.com/cd/0/inline/CkibHnp7-9WqDu5RFXDq7g2OsO0Fsm1G6MHVbt6bw09Gxgp2h86PAmLQuzE8UAsGMuflF94sqLqqMgRazvDeyPnUexvDjUc4_Q3pZTNGP6_zfuyNEcVLtFO4dAEJn5Pjh3-Mn6ek2FtCTuq71RQNWOfJ/file# [following]
--2025-02-21 13:31:23--  https://uca9c5d81c8706f19986c45534c3.

In [3]:
#First we import necessary library such as math, nltk, bigram, and collections.
import math
import nltk
import io
import random
from random import shuffle
from nltk import bigrams, trigrams
from collections import Counter, defaultdict
random.seed(999)

BEST2010 is a free Thai NLP dataset by NECTEC usually used as a standard benchmark for various NLP tasks including language modeling. It is separated into 4 domains including article, encyclopedia, news, and novel. The data is already  tokenized using '|' as a separator.

For example,

à¸•à¸²à¸¡|à¸—à¸µà¹ˆ|à¸™à¸²à¸‡à¸›à¸£à¸°à¸™à¸­à¸¡ à¸—à¸­à¸‡à¸ˆà¸±à¸™à¸—à¸£à¹Œ| |à¸à¸±à¸š| |à¸”.à¸Š.à¸à¸´à¸•à¸•à¸´à¸žà¸‡à¸©à¹Œ à¹à¸«à¸¥à¸¡à¸œà¸±à¸à¹à¸§à¹ˆà¸™| |à¹à¸¥à¸°| |à¸”.à¸.à¸à¸²à¸à¸ˆà¸™à¸² à¸à¸£à¸­à¸‡à¹à¸à¹‰à¸§| |à¸›à¹ˆà¸§à¸¢|à¸ªà¸‡à¸ªà¸±à¸¢|à¸•à¸´à¸”|à¹€à¸Šà¸·à¹‰à¸­|à¹„à¸‚à¹‰|à¸‚à¸“à¸°|à¸™à¸µà¹‰|à¸¢à¸±à¸‡|à¹„à¸¡à¹ˆ|à¸”à¸µ|à¸‚à¸¶à¹‰à¸™|

In [4]:
total_word_count = 0
best2010 = []
with open('BEST2010/news.txt','r',encoding='utf-8') as f:
  for i,line in enumerate(f):
    line=line.strip()[:-1] #remove the trailing |
    total_word_count += len(line.split("|"))
    best2010.append(line)

In [5]:
#For simplicity, we assumes that each line is a sentence.
print (f'Total sentences in BEST2010 news dataset :\t{len(best2010)}')
print (f'Total word counts in BEST2010 news dataset :\t{total_word_count}')

Total sentences in BEST2010 news dataset :	30969
Total word counts in BEST2010 news dataset :	1660190


We separate the input into 2 sets, train and test data with 70:30 ratio

In [6]:
sentences = best2010
# The data is separated to train and test set with 70:30 ratio.
train = sentences[:int(len(sentences)*0.7)]
test = sentences[int(len(sentences)*0.7):]

#Training data
train_word_count =0
for line in train:
    for word in line.split('|'):
        train_word_count+=1
print ('Total sentences in BEST2010 news training dataset :\t'+ str(len(train)))
print ('Total word counts in BEST2010 news training dataset :\t'+ str(train_word_count))

Total sentences in BEST2010 news training dataset :	21678
Total word counts in BEST2010 news training dataset :	1042797


Here we load the data from Wikipedia which is also already tokenized. It will be used for answering questions in MyCourseville.

In [7]:
import pandas as pd
wiki_data = pd.read_csv("tokenized_wiki_sample.csv")

## Data Preprocessing

Before training any language models, the first step we always do is process the data into the format suited for the LM.

For this exercise, we will use NLTK to help process our data.

In [8]:
from nltk.lm.preprocessing import pad_both_ends, flatten
from nltk.lm.vocabulary import Vocabulary
from nltk import ngrams

We begin by "tokenizing" our training set. Note that the data is already tokenized so we can just split it.

In [9]:
tokenized_train = [["<s>"] + t.split("|") + ["</s>"] for t in train] # "tokenize" and pad each sentence

Next we create a vocabulary with the ```Vocabulary``` class from NLTK. It accepts a list of tokens so we flatten our sentences into one long sentence first.







In [10]:
flat_tokens = list(flatten(tokenized_train)) #join all sentences into one long sentence
vocab = Vocabulary(flat_tokens, unk_cutoff=3) #Words with frequency **below** 3 (not exactly 3) will not be considered in our vocab and will be converted to <UNK>.

Then we replace low frequency words.

Now *each* sentence is going to look something like this:
\["\<s\>", "hello", "my", "name", "is", "\<UNK\>", "\</s\>" \]

In [11]:
tokenized_train = [[token if token in vocab else "<UNK>" for token in sentence] for sentence in tokenized_train]

Finally, we do the same for the test set and the wiki dataset.

In [12]:
tokenized_test = [t.split("|") for t in test]
tokenized_test = [[token if token in vocab else "<UNK>" for token in sentence] for sentence in tokenized_test]

tokenized_wiki_test = [t.split("|") for t in wiki_data['tokenized'].tolist()]
tokenized_wiki_test = [[token if token in vocab else "<UNK>" for token in sentence] for sentence in tokenized_wiki_test]

# Unigram

In this section, we will demonstrate how to build a unigram language model <br>
**Important note:** <br>
**\<s\>** = sentence start symbol <br>
**\</s\>** = sentence end symbol

# VERY IMPORTANT:
- In this notebook, we will *not* default the unknown token probability to ```1/len(vocab)``` but instead will treat it as a normal word and let the model learn its probability so that we can compare our results to NLTK.
- **Also make sure that the code in this notebook can be executed without any problem. If we find that you used NLTK to answer questions in MyCourseVille and did not finish the assignment, you will receive a grade of 0 for this assignment.**

In [13]:
class UnigramModel():
  def __init__(self, data, vocab):
    self.unigram_count = defaultdict(lambda: 0.0)
    self.word_count = 0
    self.vocab = vocab
    for sentence in data:
        for w in sentence: #[(word1, ), (word2, ), (word3, )...]
          w = w[0]
          self.unigram_count[w] +=1.0
          self.word_count+=1

  def __getitem__(self, w):
    w = w[0]  #[(word1, ), (word2, ), (word3, )...]
    if w in self.vocab:
      return self.unigram_count[w]/(self.word_count)
    else:
      return self.unigram_count["<UNK>"]/(self.word_count)

In [14]:
train_unigrams = [list(ngrams(sent, n=1)) for sent in tokenized_train] #creating the unigrams by setting n=1
model = UnigramModel(train_unigrams, vocab)

In [15]:
def getLnValue(x):
      if x == 0:
        return -math.inf
      return math.log(x)

In [16]:
#problability of 'à¸™à¸²à¸¢à¸'
print(getLnValue(model[('à¸™à¸²à¸¢à¸',)]))

#for example, problability of 'à¸™à¸²à¸¢à¸à¸£à¸±à¸à¸¡à¸™à¸•à¸£à¸µ' which is an unknown word is equal to
print(getLnValue(model[('à¸™à¸²à¸¢à¸à¸£à¸±à¸à¸¡à¸™à¸•à¸£à¸µ',)]))

#problability of 'à¸™à¸²à¸¢à¸' 'à¹„à¸”à¹‰' 'à¹ƒà¸«à¹‰' 'à¸ªà¸±à¸¡à¸ à¸²à¸©à¸“à¹Œ' 'à¸à¸±à¸š' 'à¸ªà¸·à¹ˆà¸­'
prob = getLnValue(model[('à¸™à¸²à¸¢à¸',)])+getLnValue(model[('à¹„à¸”à¹‰',)])+ getLnValue(model[('à¹ƒà¸«à¹‰',)])+getLnValue(model[('à¸ªà¸±à¸¡à¸ à¸²à¸©à¸“à¹Œ',)])+getLnValue(model[('à¸à¸±à¸š',)])+getLnValue(model[('à¸ªà¸·à¹ˆà¸­',)])+getLnValue(model[('</s>',)])
print ('Problability of a sentence', math.exp(prob))

-6.571687039690381
-3.952132570275872
Problability of a sentence 4.877889285183675e-18


# Perplexity

In order to compare language model we need to calculate perplexity. In this task you should write a perplexity calculation code for the unigram model. The result perplexity should be around 448.90 and
392.74 on train and test data.

## TODO #1 Calculate perplexity

In [17]:
def getLnValue(x):
    return math.log(x)

def calculate_sentence_ln_prob(sentence, model):
    prob = 0
    for word in sentence:
        prob += getLnValue(model[word])
    return prob

def perplexity(test,model):
    ln_prob = 0
    word_count = 0
    for sentence in test:
        ln_prob += calculate_sentence_ln_prob(sentence, model)
        word_count += len(sentence)
    return round(math.exp(-ln_prob/word_count),2)

In [18]:
test_unigrams = [list(ngrams(sent, n=1)) for sent in tokenized_test]

In [19]:
print(perplexity(train_unigrams,model))
print(perplexity(test_unigrams,model))

448.9
392.74


## Q1 MCV
Calculate the perplexity of the model on the wiki test set and answer in MyCourseVille

In [20]:
wiki_test_unigrams = [list(ngrams(sent, n=1)) for sent in tokenized_wiki_test]

In [21]:
print(perplexity([list(flatten(wiki_test_unigrams))], model))

485.73


# Bigram

Next, you will create a better language model than a unigram (which is not much to compare with). But first, it is very tedious to count every pair of words that occur in our corpus by ourselves. Lucky for us, nltk provides us a simple library which will simplify the process.

In [22]:
#example of nltk usage for bigram
sentence = 'I always search google for an answer .'
padded_sentence = list(pad_both_ends(sentence.split(), n=2))

print('This is how nltk generate bigram.')
for w1,w2 in bigrams(padded_sentence):
    print(w1,w2)
print('\n<s> and </s> are used as a start and end of sentence symbol. respectively.')

This is how nltk generate bigram.
<s> I
I always
always search
search google
google for
for an
an answer
answer .
. </s>

<s> and </s> are used as a start and end of sentence symbol. respectively.


Now, you should be able to implement a bigram model by yourself. Also, you must create a new perplexity calculation for bigram. The result perplexity should be around 56.46 and inf on train and test data.

## TODO #3 Write Bigram Model

In [23]:
class BigramModel():
    def __init__(self, data, vocab):
        self.bigram_count = defaultdict(lambda: 0.0)
        self.unigram_count = defaultdict(lambda: 0.0)
        self.vocab = vocab

        for sentence in data:
            for word in sentence:
                self.bigram_count[word] += 1.0
                self.unigram_count[word[0]] += 1.0
            self.unigram_count[sentence[-1][-1]] += 1.0


    def __getitem__(self, bigram):
        w1, w2 = bigram
        return self.bigram_count[(w1, w2)]/self.unigram_count[w1]

## TODO #4 Write Perplexity for Bigram Model

Sum perplexity score at a sentence level, instead of word level

In [24]:
def getLnValue(x):
    if x is None or x == 0:  # Check for None or 0
        return -math.inf
    return math.log(x)

def calculate_bigram_ln_prob(sentence, model):
    prob = 0
    cnt = 0
    for bigram in sentence:
        prob += getLnValue(model[bigram])
        cnt += 1
    return prob,cnt

def perplexity(bigram_data, model):
    ln_prob = 0
    bigram_count = 0
    for sentence in bigram_data:
        ln_prob += calculate_bigram_ln_prob(sentence, model)[0]
        bigram_count += calculate_bigram_ln_prob(sentence, model)[1]

    return round(math.exp(-ln_prob / bigram_count), 2)

In [25]:
train_bigrams = [list(ngrams(sent, n=2)) for sent in tokenized_train]
test_bigrams = [list(ngrams(sent, n=2)) for sent in tokenized_test]

In [26]:
bigram_model_scratch = BigramModel(train_bigrams, vocab)

In [27]:
print(perplexity([list(flatten(train_bigrams))], bigram_model_scratch))
print(perplexity([list(flatten(test_bigrams))[:16]], bigram_model_scratch)) #can be used to compare with nltk
print(perplexity([list(flatten(test_bigrams))], bigram_model_scratch))

56.46
24.6
inf


## Q2 MCV

In [28]:
wiki_test_bigrams = [list(ngrams(sent, n=2)) for sent in tokenized_wiki_test]

In [29]:
print(perplexity([list(flatten(wiki_test_bigrams))],bigram_model_scratch))

inf


# Smoothing

Usually any ngram models have a sparsity problem, which means it does not have every possible ngram of words in the dataset. Smoothing techniques can alleviate this problem. In this section, you will implement three basic smoothing methods laplace smoothing, interpolation for bigram, and Knesey-Ney smoothing.

## TODO #5 write Bigram with Laplace smoothing (Add-One Smoothing)

The result perplexity on training and testing should be:

    370.28, 369.16 for Laplace smoothing

In [30]:
class BigramWithLaplaceSmoothing():

  def __init__(self, data, vocab):
    self.data = data
    self.vocab = vocab
    self.bigram_count = defaultdict(lambda: 0.0)
    self.unigram_count = defaultdict(lambda: 0.0)
    self.total_word_count = 0

    for sentence in data:
      for word in sentence:
        self.bigram_count[word] += 1.0
        self.unigram_count[word[0]] += 1.0
      self.unigram_count[sentence[-1][-1]] += 1.0

  def __getitem__(self, bigram):
    return (self.bigram_count[bigram] + 1) / (self.unigram_count[bigram[0]] + len(self.vocab))

model = BigramWithLaplaceSmoothing(train_bigrams, vocab)
print(perplexity([list(flatten(train_bigrams))],model))
print(perplexity([list(flatten(test_bigrams))], model))

370.28
369.16


## Q3 MCV

In [31]:
print(perplexity([list(flatten(wiki_test_bigrams))],model))

735.03


## TODO #6 Write Bigram with Interpolation
Set the lambda value as 0.7 for bigram, 0.25 for unigram, and 0.05 for unknown word.

The result perplexity on training and testing should be:

    70.07, 102.99 for Interpolation

In [32]:
class BigramWithInterpolation():

  def __init__(self, data, vocab):
    self.data = data
    self.vocab = vocab
    self.bigram_count = defaultdict(lambda: 0.0)
    self.unigram_count = defaultdict(lambda: 0.0)
    self.total_word_count = 0

    for sentence in data:
      for word in sentence:
        self.bigram_count[word] += 1.0
        self.unigram_count[word[0]] += 1.0
        self.total_word_count += 1
      self.unigram_count[sentence[-1][-1]] += 1.0
      self.total_word_count += 1

  def __getitem__(self, bigram):
    w1, w2 = bigram
    unigram_prob = self.unigram_count[w2] / self.total_word_count if self.total_word_count > 0 else 0
    bigram_prob = self.bigram_count[bigram] / self.unigram_count[w1] if self.unigram_count[w1] > 0 else 0
    return 0.7 * bigram_prob + \
             0.25 * unigram_prob + \
             0.05 * (1 / len(self.vocab))

model = BigramWithInterpolation(train_bigrams, vocab)
print(perplexity([list(flatten(train_bigrams))],model))
print(perplexity([list(flatten(test_bigrams))], model))

70.07
102.99


## Q4 MCV

In [33]:
print(perplexity([list(flatten(wiki_test_bigrams))],model))

251.61


## Language modeling on multiple domains

Sometimes, we do not have enough data to create a language model for a new domain. In that case, we can improvised by combining several models to improve result on the new domain.

In this exercise you will try to merge two language models from news and article domains to create a language model for the encyclopedia domain.

In [34]:
# create encyclopeida data (test data)
encyclo_data=[]
with open('BEST2010/encyclopedia.txt','r',encoding='utf-8') as f:
    for i,line in enumerate(f):
        encyclo_data.append(line.strip()[:-1])

(news) First, you should try to calculate perplexity of your bigram with interpolation on encyclopedia data. The  perplexity should be around 236.33

In [35]:
def create_bigrams(data,vocab):
  tokenized_train = [["<s>"] + t.split("|") + ["</s>"] for t in data] # "tokenize" and pad each sentence
  tokenized_train = [[token if token in vocab else "<UNK>" for token in sentence] for sentence in tokenized_train]
  train_bigrams = [list(ngrams(sent, n=2)) for sent in tokenized_train]
  return train_bigrams


In [36]:
encyclopedia_bigrams = create_bigrams(encyclo_data,vocab)
model = BigramWithInterpolation(train_bigrams, vocab)

In [37]:
# 1) news only on "encyclopedia"
print(perplexity([list(flatten(encyclopedia_bigrams))], model))

236.33


## TODO #7 - Langauge Modelling on Multiple Domains
Combine news and article datasets to create another bigram model and evaluate it on the encyclopedia data.



(article) For your information, a bigram model with interpolation using article data to test on encyclopedia data has a perplexity of 218.57

In [38]:
def create_vocab(data):
  tokenized = [["<s>"] + t.split("|") + ["</s>"] for t in data]
  vocab = Vocabulary(list(flatten(tokenized)), unk_cutoff=3)
  return vocab

In [39]:
# 2) article only on "encyclopedia"
best2010_article=[]
with open('BEST2010/article.txt','r',encoding='utf-8') as f:
    for i,line in enumerate(f):
        best2010_article.append(line.strip()[:-1])

combined_total_word_count = 0
for line in best2010_article:
    combined_total_word_count += len(line.split('|'))

article_vocab = create_vocab(best2010_article)
article_bigrams = create_bigrams(best2010_article,article_vocab)


model = BigramWithInterpolation(article_bigrams, article_vocab)

In [40]:
print('Perplexity of the bigram model using article data with interpolation smoothing on encyclopedia test data',perplexity([list(flatten(encyclopedia_bigrams))], model))

Perplexity of the bigram model using article data with interpolation smoothing on encyclopedia test data 218.57


In [41]:
# 3) train on news + article, test on "encyclopedia"
best2010_article_and_news = best2010_article.copy()
with open('BEST2010/news.txt','r',encoding='utf-8') as f:
    for i,line in enumerate(f):
        best2010_article_and_news.append(line.strip()[:-1])

combined_vocab =  create_vocab(best2010_article_and_news)
combined_bigrams = create_bigrams(best2010_article_and_news,combined_vocab)


combined_model = BigramWithInterpolation(combined_bigrams, combined_vocab)
print('Perplexity of the combined Bigram model with interpolation smoothing on encyclopedia test data',perplexity([list(flatten(encyclopedia_bigrams))], combined_model))

Perplexity of the combined Bigram model with interpolation smoothing on encyclopedia test data 251.73


## TODO #8 - Kneser-ney on "News"

<!-- Reimplement equation 4.33 in SLP textbook (https://lagunita.stanford.edu/c4x/Engineering/CS-224N/asset/slp4.pdf) -->

Implement Bigram Knerser-ney LM. The result perplexity should be around 65.81, 93.21 on train and test data. Be careful not to mix up vocab from the above section!


In [42]:
print(vocab)

<Vocabulary with cutoff=3 unk_label='<UNK>' and 9063 items>


In [43]:
class BigramKneserNey():
  def __init__(self, data, vocab):
    self.data = data
    self.vocab = vocab
    self.bigram_count = defaultdict(lambda: 0.0)
    self.unigram_count = defaultdict(lambda: 0.0)
    self.start = defaultdict(set)
    self.end = defaultdict(set)

    for sentence in data:
      for word in sentence:
        self.bigram_count[word] += 1
        self.unigram_count[word[0]] += 1
        self.start[word[0]].add(word[1])
        self.end[word[1]].add(word[0])
      self.unigram_count[sentence[-1][-1]] += 1.0
    self.bigram_size = len(self.bigram_count)

  def __getitem__(self, bigram):
    w1, w2 = bigram
    s1 = max(self.bigram_count[bigram] - 0.75, 0) / self.unigram_count[w1]
    s2 = 0.75 * len( self.start[w1]) / self.unigram_count[w1]
    s3 = len(self.end[w2]) / self.bigram_size
    return s1 + (s2 * s3)

model = BigramKneserNey(train_bigrams, vocab)
print(perplexity([list(flatten(train_bigrams))],model))
print(perplexity([list(flatten(train_bigrams))[:1000]],model)) #can be used to compare with nltk
print(perplexity([list(flatten(test_bigrams))[:1000]], model)) #can be used to compare with nltk
print(perplexity([list(flatten(test_bigrams))], model))

65.81
50.41
90.3
93.21


## Q5 MCV

In [44]:
print(perplexity([list(flatten(wiki_test_bigrams))],model))

266.58
